# Deepfake Audio Detection - Self-Contained Google Colab Training Notebook

This notebook trains the 4 refined models (Wav2Vec2, AASIST, CQCC, CustomHybrid) locally via Cloud GPUs.

In [ ]:
!pip install -q torch torchaudio torchvision transformers librosa scikit-learn matplotlib seaborn tqdm soundfile datasets scipy accelerate huggingface_hub

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Model


# ============================================================
# 1. Wav2Vec2 Detector (Self-supervised Transformer Baseline)
# ============================================================
class AttentivePooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return torch.sum(w * x, dim=1)

class Wav2Vec2SpoofDetector(nn.Module):
    def __init__(self, num_classes=2, model_name="facebook/wav2vec2-base"):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained(model_name)
        #freeze model
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        hidden = self.wav2vec.config.hidden_size
        self.pool = AttentivePooling(hidden)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        out = self.wav2vec(x).last_hidden_state
        pooled = self.pool(out)
        return self.classifier(pooled)

# ============================================================
# 2. AASIST (SOTA Graph-based Baseline)
# ============================================================

class GraphAttention(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.attn = nn.Linear(out_dim * 2, 1)

    def forward(self, x):
        h = self.fc(x)
        B, N, D = h.shape
        h_i = h.unsqueeze(2).expand(-1, -1, N, -1)
        h_j = h.unsqueeze(1).expand(-1, N, -1, -1)
        attn_input = torch.cat([h_i, h_j], dim=-1)
        e = F.softmax(self.attn(attn_input).squeeze(-1), dim=-1)
        out = torch.matmul(e, h)
        return out

class GraphBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gat = GraphAttention(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        res = x
        x = self.gat(x)
        x = self.norm(x + res)
        return x

class AASISTDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # CNN backbone
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # Graph layers
        self.graph1 = GraphBlock(128)
        self.graph2 = GraphBlock(128)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        B, C, H, W = x.shape
        nodes = x.view(B, C, -1).permute(0, 2, 1)
        nodes = self.graph1(nodes)
        nodes = self.graph2(nodes)
        pooled = nodes.mean(dim=1)
        return self.fc(pooled)

# ============================================================
# 3. CQCC Baseline Detector (Acoustic Feature Baseline)
# ============================================================

class CQCCBaselineDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Input shape expected: (B, 1, 20, T)
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

# ============================================================
# 4. Custom Fusional Wav2Vec2 + CQCC with Cross-Attention + Graph
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=2000):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, dim))

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1)]

class BidirectionalCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn1 = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn2 = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)

    def forward(self, x1, x2):
        # x1 attends to x2
        q1 = self.norm_q(x1)
        k2 = self.norm_kv(x2)
        v2 = k2
        out1, _ = self.attn1(q1, k2, v2)

        # x2 attends to x1
        q2 = self.norm_q(x2)
        k1 = self.norm_kv(x1)
        v1 = k1
        out2, _ = self.attn2(q2, k1, v1)
        return out1, out2

def align_sequences(x, target_len):
    """Linear interpolation to match sequence lengths"""
    x = x.transpose(1, 2)
    x = F.interpolate(x, size=target_len, mode='linear', align_corners=False)
    return x.transpose(1, 2)

class ImprovedWav2Vec2CQCCDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Wav2Vec2
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        # Freeze the Wav2Vec2 layer so it acts purely as a feature extractor
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        dim = self.wav2vec.config.hidden_size

        # CQCC encoder
        self.cqcc_conv = nn.Sequential(
            nn.Conv1d(20, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )

        # Positional Encoding
        self.pos_enc = PositionalEncoding(dim)

        # Bidirectional Cross Attention
        self.cross_attn = BidirectionalCrossAttention(dim)

        # True Graph Transformer Backend (using GAT blocks from AASIST)
        self.graph_layers = nn.ModuleList([
            GraphBlock(dim) for _ in range(3)
        ])

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, wav, cqcc):
        if wav.dim() == 3:
            wav = wav.squeeze(1)

        # Wav2Vec2 features
        w2v = self.wav2vec(wav).last_hidden_state  # (B, T_w, D)

        # CQCC features
        if cqcc.dim() == 4:
            cqcc = cqcc.squeeze(1)
        cqcc_feat = self.cqcc_conv(cqcc).transpose(1, 2)  # (B, T_c, D)

        # Align lengths
        cqcc_feat = align_sequences(cqcc_feat, w2v.size(1))

        # Add positional encoding
        w2v = self.pos_enc(w2v)
        cqcc_feat = self.pos_enc(cqcc_feat)

        # Cross attention (bidirectional)
        f1, f2 = self.cross_attn(cqcc_feat, w2v)
        fused = f1 + f2

        # Graph Transformer processing on node sequences
        x = fused
        for layer in self.graph_layers:
            x = layer(x)

        # Global average pooling on the nodes
        pooled = x.mean(dim=1)

        return self.classifier(pooled)

## 2. Define Dataset and Data Utilities

In [ ]:
import os
import torch
import torchaudio
import numpy as np
import librosa
from scipy.fftpack import dct
import glob
from tqdm.notebook import tqdm

def precompute_features(data_dir="/content/drive/MyDrive/MLAAD-tiny", n_mels=60):
    save_dir = os.path.join(data_dir, "precomputed_features")
    print(f"Precomputing features to {save_dir}... This will take a while but will drastically speed up training!")
    
    real_path = os.path.join(data_dir, "original")
    fake_path = os.path.join(data_dir, "fake")
    
    files_and_labels = []
    
    if os.path.exists(real_path):
        for root, dirs, files in os.walk(real_path):
            for f in files:
                if f.endswith('.wav') or f.endswith('.flac'):
                    files_and_labels.append((os.path.join(root, f), 0))
                    
    if os.path.exists(fake_path):
        for root, dirs, files in os.walk(fake_path):
            for f in files:
                if f.endswith('.wav') or f.endswith('.flac'):
                    files_and_labels.append((os.path.join(root, f), 1))
                    
    mel_spectrogram = torchaudio.transforms.MelSpectrogram(
        sample_rate=16000, n_mels=n_mels, n_fft=400, hop_length=160
    )
    
    for filepath, label in tqdm(files_and_labels):
        # Calculate the relative path from the data_dir 
        # e.g., 'original/en/audio.wav'
        relative_path = os.path.relpath(filepath, data_dir)
        
        # Remove the original audio extension
        rel_path_no_ext = os.path.splitext(relative_path)[0]
        
        # Create the mirrored save path
        # e.g., '/content/drive/MyDrive/MLAAD-tiny/precomputed_features/original/en/audio.pt'
        save_path = os.path.join(save_dir, f"{rel_path_no_ext}.pt")
        
        # Ensure the subdirectories exist
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        if os.path.exists(save_path):
            continue # Skip already processed
            
        try:
            wav_np, sr = librosa.load(filepath, sr=16000, mono=True)
            wav = torch.from_numpy(wav_np).unsqueeze(0).float()
            
            mel = mel_spectrogram(wav)
            mel = torchaudio.transforms.AmplitudeToDB()(mel)
            
            # CQT
            cqt = np.abs(librosa.cqt(wav_np, sr=16000, n_bins=n_mels, hop_length=160, fmin=librosa.note_to_hz('C1')))
            log_power = librosa.amplitude_to_db(cqt, ref=np.max)
            cqcc = dct(log_power, type=2, axis=0, norm='ortho')[:20]
            cqcc = torch.from_numpy(cqcc).unsqueeze(0).float()
            
            # We save everything in a PyTorch dictionary. 
            # Notice that we store 'original_path' along with the tensors.
            torch.save({
                'mel': mel,
                'wav': wav,
                'cqcc': cqcc,
                'label': label,
                'original_path': filepath
            }, save_path)
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

# Run precomputation
precompute_features()


In [3]:
import os
import torch
import torchaudio
import numpy as np
from torch.utils.data import Dataset
import glob

class AudioDataset(Dataset):
    def __init__(self, data_dir="/content/drive/MyDrive/MLAAD-tiny/precomputed_features", augment=False):
        # Read from both original and fake subfolders
        self.files = glob.glob(os.path.join(data_dir, "**", "*.pt"), recursive=True)
        self.augment = augment
        
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=20)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # We load the dictionary which contains our saved features + the original path string
        data = torch.load(self.files[idx])
        
        mel = data['mel']
        wav = data['wav']
        cqcc = data['cqcc']
        label = data['label']
        # The original path is also saved inside the dictionary:
        # original_path = data['original_path']

        # Augmentation on raw audio (Data Augmentation for generalizability)
        if self.augment and np.random.rand() < 0.3:
            noise = torch.randn_like(wav) * 0.002
            wav = wav + noise
            
        # Spectrogram Augmentation
        if self.augment and torch.rand(1).item() < 0.5:
            mel = self.freq_mask(mel)
            mel = self.time_mask(mel)

        return mel, wav, cqcc, label

def collate_variable_length(batch):
    mels, wavs, cqccs, labels = zip(*batch)
    labels = torch.tensor(labels)

    # ---------- MEL ----------
    max_mel_time = max(m.shape[-1] for m in mels)
    mels_padded = []
    for m in mels:
        if m.shape[-1] < max_mel_time:
            pad = max_mel_time - m.shape[-1]
            m = torch.nn.functional.pad(m, (0, pad))
        mels_padded.append(m)
    mels = torch.stack(mels_padded, dim=0)

    # ---------- WAVE ----------
    max_wav_len = max(w.shape[-1] for w in wavs)
    wavs_padded = []
    for w in wavs:
        if w.shape[-1] < max_wav_len:
            pad = max_wav_len - w.shape[-1]
            w = torch.nn.functional.pad(w, (0, pad))
        wavs_padded.append(w)
    wavs = torch.stack(wavs_padded, dim=0)

    # ---------- CQCC ----------
    max_cqcc_len = max(c.shape[-1] for c in cqccs)
    cqccs_padded = []
    for c in cqccs:
        if c.shape[-1] < max_cqcc_len:
            pad = max_cqcc_len - c.shape[-1]
            c = torch.nn.functional.pad(c, (0, pad))
        cqccs_padded.append(c)
    cqccs = torch.stack(cqccs_padded, dim=0)

    return mels, wavs, cqccs, labels


In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import random


def train_model(model, dataloader, criterion, optimizer, epochs=5, input_type='mel', device=None):
    """Train model on dataset."""

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.train()
    loss_history = []

    for epoch in range(epochs):

        epoch_loss = 0
        correct = 0
        total = 0

        for batch in dataloader:

            if input_type == 'mel':
                outputs = model(batch[0].to(device))
            elif input_type == 'wav':
                outputs = model(batch[1].to(device))
            elif input_type == 'cqcc':
                outputs = model(batch[2].to(device))
            elif input_type == 'wav_and_cqcc':
                outputs = model(batch[1].to(device), batch[2].to(device))
            else:
                raise ValueError("invalid input_type")

            labels = batch[-1].to(device)

            optimizer.zero_grad()

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        acc = 100 * correct / total if total > 0 else 0
        avg_loss = epoch_loss / len(dataloader)

        loss_history.append(avg_loss)

        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    return loss_history


def evaluate_model(model, dataloader, input_type='mel', device=None):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in dataloader:

            if input_type == 'mel':
                outputs = model(batch[0].to(device))
            elif input_type == 'wav':
                outputs = model(batch[1].to(device))
            elif input_type == 'cqcc':
                outputs = model(batch[2].to(device))
            elif input_type == 'wav_and_cqcc':
                outputs = model(batch[1].to(device), batch[2].to(device))

            labels = batch[-1].to(device)

            probs = torch.softmax(outputs, dim=1)[:, 1]

            all_labels.extend(labels.tolist())
            all_probs.extend(probs.tolist())

    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    # ------------------
    # EER (Equal Error Rate)
    # ------------------
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute(fnr - fpr))
    eer = fpr[eer_index]

    # ------------------
    # minDCF (Minimum Detection Cost Function)
    # Parameters for ASVSpoof/NIST standards
    # ------------------
    P_spoof = 0.05  # Prior probability of encountering a deepfake
    C_miss = 10     # Cost of missing a deepfake (False Negative)
    C_fa = 1        # Cost of falsely rejecting a real voice (False Positive)

    # DCF = C_miss * P_miss * P_spoof + C_fa * P_fa * (1 - P_spoof)
    # Where P_miss = fnr, P_fa = fpr
    dcf_array = C_miss * fnr * P_spoof + C_fa * fpr * (1 - P_spoof)
    min_dcf = np.min(dcf_array)

    return fpr, tpr, roc_auc, eer, min_dcf

# Set up and seed

In [5]:
import os, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Dataset + Loader

In [9]:
print("Loading Dataset with Augmentation...")

dataset = AudioDataset(augment=True)

subset_size = 3000  # keep small for GPU safety
dataset = torch.utils.data.Subset(dataset, range(subset_size))

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

split_generator = torch.Generator().manual_seed(SEED)
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=split_generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_variable_length,
    num_workers = 2,
    pin_memory = True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_variable_length
)

models_dir = "/content/drive/MyDrive/models"
os.makedirs(models_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()

Loading Dataset with Augmentation...


# TRAIN EACH MODEL IN SEPARATE CELLS

In [10]:
print("\n--- Training Wav2Vec2 Baseline ---")

wav2vec_model = Wav2Vec2SpoofDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(wav2vec_model.parameters(), lr=1e-4)

train_model(
    wav2vec_model,
    train_loader,
    criterion,
    optimizer,
    input_type='wav',
    device=device
)

torch.save(wav2vec_model.state_dict(), os.path.join(models_dir, "wav2vec2.pth"))


--- Training Wav2Vec2 Baseline ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=367
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.81 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.25 GiB is free. Including non-PyTorch memory, this process has 11.31 GiB memory in use. Of the allocated memory 6.13 GiB is allocated by PyTorch, and 5.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print("\n--- Training AASIST Baseline ---")

aasist_model = AASISTDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(aasist_model.parameters(), lr=5e-4)

train_model(
    aasist_model,
    train_loader,
    criterion,
    optimizer,
    input_type='mel',
    device=device
)

torch.save(aasist_model.state_dict(), os.path.join(models_dir, "aasist.pth"))

In [11]:
print("\n--- Training CQCC Baseline ---")

cqcc_model = CQCCBaselineDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(cqcc_model.parameters(), lr=1e-3)

train_model(
    cqcc_model,
    train_loader,
    criterion,
    optimizer,
    input_type='cqcc',
    device=device
)

torch.save(cqcc_model.state_dict(), os.path.join(models_dir, "cqcc_baseline.pth"))


--- Training CQCC Baseline ---


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=367
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=482
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=448
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=468
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=383
  warnings.warn(


Epoch 1/5 | Loss: 0.1814 | Acc: 98.67%


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=448
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=383
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=468
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=367
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=482
  warnings.warn(


Epoch 2/5 | Loss: 0.0224 | Acc: 100.00%


KeyboardInterrupt: 

In [ ]:
print("\n--- Training Custom Fusion Detector ---")

custom_model = ImprovedWav2Vec2CQCCDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(custom_model.parameters(), lr=1e-4)

train_model(
    custom_model,
    train_loader,
    criterion,
    optimizer,
    input_type='wav_and_cqcc',
    device=device
)

torch.save(custom_model.state_dict(), os.path.join(models_dir, "custom_hybrid.pth"))

# Evaluation

In [ ]:
print("\n--- Evaluating Models ---")

def load_model(model_class, path):
    model = model_class(num_classes=2).to(device)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

models_to_eval = [
    ("Wav2Vec2", load_model(Wav2Vec2SpoofDetector, os.path.join(models_dir, "wav2vec2.pth")), 'wav'),
    ("AASIST", load_model(AASISTDetector, os.path.join(models_dir, "aasist.pth")), 'mel'),
    ("CQCC", load_model(CQCCBaselineDetector, os.path.join(models_dir, "cqcc_baseline.pth")), 'cqcc'),
    ("Fusion", load_model(ImprovedWav2Vec2CQCCDetector, os.path.join(models_dir, "custom_hybrid.pth")), 'wav_and_cqcc')
]

evals = []

for name, model, inp in models_to_eval:
    fpr, tpr, auc_val, eer, min_dcf = evaluate_model(
        model,
        test_loader,
        input_type=inp,
        device=device
    )

    evals.append((name, fpr, tpr, auc_val, eer, min_dcf))
    print(f"{name} | AUC={auc_val:.3f} | EER={eer*100:.2f}%")